# 🌧️ ANUGA Overland Flow Simulation with IMERG Rainfall Data

## Overview
This notebook simulates overland flow using real rainfall data from IMERG (Integrated Multi-satellitE Retrievals for GPM). The rainfall data is provided as time-series at discrete grid points defined by latitude/longitude coordinates.

## Key Features:
- **Real Rainfall Data**: Uses IMERG hourly precipitation data from 2022
- **Cell-Based Rainfall Distribution**: Each rainfall cell has constant value = average of 4 corner grid points
- **No Interpolation**: Preserves original low rainfall values without artificial smoothing
- **GPU Acceleration**: Leverages CUDA/CuPy for 5-10x speedup
- **UTM Coordinate Transformation**: Converts WGS84 lat/lon to UTM Zone 43N
- **Time-varying Rainfall**: Applies rainfall that changes temporally (hourly updates)

## Rainfall Method:
Given the coarse spatial resolution of IMERG data (~10-11 km grid spacing) and already low rainfall values (mm/hr), interpolation is not appropriate. Instead, this notebook uses a **cell-based approach**:
- Each domain triangle is assigned to a rainfall cell
- Cell value = average of 4 corner grid points
- Constant rainfall within each cell for 1 hour duration
- This preserves the original data characteristics without artificial smoothing

## Data Sources:
- **DEM**: `DEM_UTM_EPSG32643.tif` (UTM Zone 43N)
- **Rainfall**: `IMERG_hourly_2022_ROI.csv` (WGS84 lat/lon grid)
- **Domain**: Derived from bounding box

## Requirements:
- ANUGA with CUDA support
- CuPy (for GPU acceleration)
- GDAL, pandas, numpy, scipy, matplotlib
- NVIDIA GPU with CUDA

## 📦 Import Libraries and Check GPU

In [ ]:
# Core libraries
import anuga
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from scipy.spatial import cKDTree
import os
import sys
import time
from pathlib import Path

# Geospatial libraries
from osgeo import gdal, osr
from shapely import wkt
from pyproj import Transformer, CRS

# Check for GPU support
GPU_AVAILABLE = False
try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("✓ CuPy found - GPU acceleration ENABLED")
    print(f"  GPU Device: {cp.cuda.Device()}")
    print(f"  CuPy version: {cp.__version__}")
    
    # Get GPU info
    mempool = cp.get_default_memory_pool()
    mem_info = cp.cuda.Device().mem_info
    print(f"  GPU Memory Available: {mem_info[0]/1e9:.2f} GB free / {mem_info[1]/1e9:.2f} GB total")
except ImportError:
    print("✗ CuPy not found - Running on CPU only")
    print("  To enable GPU: pip install cupy-cuda11x (or cupy-cuda12x)")

print(f"\nANUGA version: {anuga.__version__ if hasattr(anuga, '__version__') else 'Unknown'}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## ⚙️ Configuration

In [ ]:
class RainfallSimulationConfig:
    """Configuration for rainfall-based overland flow simulation."""
    def __init__(self):
        # Input files
        self.dem_file = "filled_smoothed_topography_EPSG32643.tif"  # DEM in UTM Zone 43N
        self.rainfall_csv = "IMERG_hourly_2022_ROI.csv"  # IMERG data
        self.bounding_box_csv = "bounder.csv"  # Domain boundary
        
        # Coordinate system
        self.target_epsg = 32643  # UTM Zone 43N
        self.source_epsg = 4326   # WGS84 (lat/lon)
        
        # Domain parameters
        self.maximum_triangle_area = 17500  # m² (mesh resolution)
        self.buffer_distance = -5000.0  # meters (inward buffer)
        
        # Physical parameters
        self.friction_coefficient = 0.0175  # Manning's n
        self.minimum_storable_height = 1e-8 # meters
        
        # Simulation time parameters
        # Parse rainfall data to determine simulation period
        self.start_date = None  # Will be set from data
        self.end_date = None    # Will be set from data
        self.simulation_days = 60  # Simulate first 7 days (can be adjusted)
        self.output_interval_hours = 6.0  # Save results every hour
        
        # GPU and performance
        self.use_gpu = GPU_AVAILABLE
        self.save_checkpoints = True
        self.checkpoint_interval_hours = 24.0 * 5
          # Daily checkpoints
        
        # Output
        self.output_dir = 'outputs'
        self.simulation_name = 'rainfall_simulation'
        
config = RainfallSimulationConfig()
print("✓ Configuration loaded")
print(f"  DEM: {config.dem_file}")
print(f"  Rainfall data: {config.rainfall_csv}")
print(f"  Target EPSG: {config.target_epsg}")
print(f"  GPU enabled: {config.use_gpu}")

## 📊 Load and Process Rainfall Data

In [ ]:
print("\n" + "="*70)
print("LOADING IMERG RAINFALL DATA")
print("="*70)

# Load rainfall CSV
print(f"\nReading {config.rainfall_csv}...")
rain_df = pd.read_csv(config.rainfall_csv)

print(f"✓ Loaded {len(rain_df):,} records")
print(f"\nColumns: {list(rain_df.columns)}")
print(f"\nFirst few rows:")
print(rain_df.head())

# Parse time column (format: DD-MM-YYYY HH:MM)
print("\nParsing timestamps...")
rain_df['datetime'] = pd.to_datetime(rain_df['time'], format='%d-%m-%Y %H:%M')

# Get unique timestamps and spatial points
unique_times = sorted(rain_df['datetime'].unique())
unique_lats = sorted(rain_df['lat'].unique())
unique_lons = sorted(rain_df['long'].unique())

print(f"✓ Parsed {len(unique_times)} unique timestamps")
print(f"  Time range: {unique_times[0]} to {unique_times[-1]}")
print(f"  Duration: {(unique_times[-1] - unique_times[0]).days} days, {(unique_times[-1] - unique_times[0]).seconds//3600} hours")

print(f"\n✓ Spatial grid: {len(unique_lats)} latitudes × {len(unique_lons)} longitudes")
print(f"  Latitude range: {min(unique_lats):.3f}° to {max(unique_lats):.3f}°")
print(f"  Longitude range: {min(unique_lons):.3f}° to {max(unique_lons):.3f}°")

# Statistics
print(f"\n✓ Precipitation statistics:")
print(f"  Min: {rain_df['precipitation'].min():.6f} mm/hr")
print(f"  Max: {rain_df['precipitation'].max():.6f} mm/hr")
print(f"  Mean: {rain_df['precipitation'].mean():.6f} mm/hr")
print(f"  Non-zero values: {(rain_df['precipitation'] > 0).sum():,} ({(rain_df['precipitation'] > 0).sum()/len(rain_df)*100:.1f}%)")

# Set simulation period
config.start_date = unique_times[0]
config.end_date = min(unique_times[0] + timedelta(days=config.simulation_days), unique_times[-1])

print(f"\n✓ Simulation period: {config.start_date} to {config.end_date}")
print(f"  Duration: {(config.end_date - config.start_date).days} days")

## 🗺️ Transform Rainfall Coordinates to UTM

In [ ]:
print("\n" + "="*70)
print("COORDINATE TRANSFORMATION: WGS84 → UTM Zone 43N")
print("="*70)

# Create transformer from WGS84 to UTM Zone 43N
print(f"\nCreating coordinate transformer...")
print(f"  Source: EPSG:{config.source_epsg} (WGS84 lat/lon)")
print(f"  Target: EPSG:{config.target_epsg} (UTM Zone 43N)")

transformer = Transformer.from_crs(
    CRS.from_epsg(config.source_epsg),
    CRS.from_epsg(config.target_epsg),
    always_xy=True
)

# Transform coordinates
print("\nTransforming rainfall grid coordinates...")
rain_df['easting'], rain_df['northing'] = transformer.transform(
    rain_df['long'].values,
    rain_df['lat'].values
)

print(f"✓ Coordinates transformed")
print(f"\nUTM coordinates:")
print(f"  Easting range: {rain_df['easting'].min():.2f} to {rain_df['easting'].max():.2f} m")
print(f"  Northing range: {rain_df['northing'].min():.2f} to {rain_df['northing'].max():.2f} m")

# Show example transformation
print(f"\nExample transformation:")
sample = rain_df.iloc[0]
print(f"  Lat/Lon: ({sample['lat']:.6f}°, {sample['long']:.6f}°)")
print(f"  UTM: ({sample['easting']:.2f} m E, {sample['northing']:.2f} m N)")

## 🏗️ Setup ANUGA Domain

In [ ]:
def parse_qgis_bounding_box(csv_path, buffer_distance=-5000.0):
    """Parse QGIS bounding box from CSV with optional buffering."""
    df = pd.read_csv(csv_path)
    wkt_string = df['WKT'].iloc[0]
    polygon_geom = wkt.loads(wkt_string)
    
    if buffer_distance != 0:
        polygon_geom = polygon_geom.buffer(buffer_distance)
    
    coords_tuples = list(polygon_geom.exterior.coords)
    anuga_polygon = [list(pt) for pt in coords_tuples]
    
    if anuga_polygon[0] == anuga_polygon[-1]:
        anuga_polygon.pop()
    
    return anuga_polygon

print("\n" + "="*70)
print("CREATING ANUGA DOMAIN")
print("="*70)

# Load boundary polygon
print(f"\nLoading boundary from {config.bounding_box_csv}...")
bounding_polygon = parse_qgis_bounding_box(config.bounding_box_csv, config.buffer_distance)
print(f"✓ Parsed {len(bounding_polygon)} vertices")

# Create mesh
print(f"\nCreating computational mesh (max triangle area: {config.maximum_triangle_area} m²)...")
num_segments = len(bounding_polygon)
tags = {'exterior': list(range(num_segments))}

start_time = time.time()
domain = anuga.create_domain_from_regions(
    bounding_polygon,
    boundary_tags=tags,
    maximum_triangle_area=config.maximum_triangle_area
)
mesh_time = time.time() - start_time

print(f"✓ Domain created in {mesh_time:.2f}s")
print(f"  Triangles: {domain.get_number_of_triangles():,}")
print(f"  Vertices: {domain.get_number_of_nodes():,}")
print(f"  Area: {domain.get_area()/1e6:.2f} km²")

# Set coordinate reference
domain.geo_reference.set_zone(43)  # UTM Zone 43N
domain.geo_reference.set_hemisphere('northern')
print(f"✓ Set UTM Zone 43N (Northern Hemisphere)")

# Configure flow algorithm for GPU
domain.set_flow_algorithm('DE0')
domain.set_low_froude(0)
print(f"✓ Flow algorithm: DE0 (GPU compatible)")

## 🏔️ Load Topography

In [ ]:
print("\n" + "="*70)
print("LOADING TOPOGRAPHY")
print("="*70)

print(f"\nLoading DEM from {config.dem_file}...")
domain.get_quantity('elevation').set_values_from_tif_file(config.dem_file)
elev = domain.get_quantity('elevation')

print(f"✓ Elevation loaded")
print(f"  Min elevation: {elev.get_minimum_value():.2f} m")
print(f"  Max elevation: {elev.get_maximum_value():.2f} m")
print(f"  Mean elevation: {np.mean(elev.centroid_values):.2f} m")

# Set initial conditions
print(f"\nSetting initial conditions...")
domain.set_quantity('friction', config.friction_coefficient)
domain.set_quantity('stage', expression='elevation')  # Dry bed
print(f"✓ Friction coefficient: {config.friction_coefficient}")
print(f"✓ Initial condition: Dry bed (stage = elevation)")

# Set boundary conditions
print(f"\nSetting boundary conditions...")
Bo = anuga.Dirichlet_boundary([-10.0, 0.0, 0.0])  # Outflow
domain.set_boundary({'exterior': Bo})
print(f"✓ Boundary: Dirichlet outflow on all exterior boundaries")

# Configure domain
domain.set_minimum_storable_height(config.minimum_storable_height)
domain.set_name(config.simulation_name)
domain.set_datadir(config.output_dir)
os.makedirs(config.output_dir, exist_ok=True)

# Enable .sww file storage
domain.set_store(True)
domain.set_store_vertices_uniquely(False)
print(f"✓ SWW file storage enabled")

## 🌧️ Prepare Rainfall Operator with Cell-Based Assignment

In [ ]:
print("\n" + "="*70)
print("PREPARING RAINFALL OPERATOR (CELL-BASED APPROACH)")
print("="*70)

# Get domain centroid coordinates
centroids = domain.get_centroid_coordinates()
domain_x = centroids[:, 0]
domain_y = centroids[:, 1]

print(f"\nDomain mesh:")
print(f"  Triangles: {len(domain_x):,}")
print(f"  X range: {domain_x.min():.2f} to {domain_x.max():.2f} m")
print(f"  Y range: {domain_y.min():.2f} to {domain_y.max():.2f} m")

# Filter rainfall data for simulation period
print(f"\nFiltering rainfall data for simulation period...")
rain_sim = rain_df[
    (rain_df['datetime'] >= config.start_date) & 
    (rain_df['datetime'] <= config.end_date)
].copy()

print(f"✓ Filtered to {len(rain_sim):,} records")

# Get unique times in simulation period (as NumPy array)
sim_times = np.array(sorted(rain_sim['datetime'].unique()))
print(f"✓ {len(sim_times)} time steps")

# Convert to seconds since start (vectorized)
rain_sim['seconds'] = (rain_sim['datetime'] - config.start_date).dt.total_seconds()
time_seconds = (pd.Series(sim_times) - config.start_date).dt.total_seconds().values

print(f"\nTime steps: 0 to {time_seconds[-1]:.0f} seconds ({time_seconds[-1]/3600:.1f} hours)")

# Get unique lat/lon grid spacing (convert to NumPy arrays immediately)
unique_lats = np.array(sorted(rain_sim['lat'].unique()))
unique_lons = np.array(sorted(rain_sim['long'].unique()))
lat_spacing = np.diff(unique_lats).mean() if len(unique_lats) > 1 else 0.1
lon_spacing = np.diff(unique_lons).mean() if len(unique_lons) > 1 else 0.1

print(f"\n✓ Rainfall grid spacing:")
print(f"  Latitude spacing: {lat_spacing:.4f}° (~{lat_spacing * 111000:.0f} m)")
print(f"  Longitude spacing: {lon_spacing:.4f}° (~{lon_spacing * 111000 * np.cos(np.radians(unique_lats[0])):.0f} m)")

# Create rainfall cell structure for each timestep
# Each cell will have constant rainfall = average of 4 corner values
print(f"\nCreating cell-based rainfall structure...")
print(f"  Method: Each cell uses average of 4 corner grid points")
print(f"  Optimizing with vectorized operations...")

# OPTIMIZATION 1: Transform domain centroids ONCE (not in loop)
from pyproj import Transformer
transformer_inv = Transformer.from_crs(
    CRS.from_epsg(config.target_epsg),
    CRS.from_epsg(config.source_epsg),
    always_xy=True
)
domain_lon, domain_lat = transformer_inv.transform(domain_x, domain_y)
print(f"  ✓ Transformed {len(domain_x):,} domain points to lat/lon")

# OPTIMIZATION 2: Pre-compute cell indices for all domain points (vectorized)
lat_indices = np.searchsorted(unique_lats, domain_lat)
lon_indices = np.searchsorted(unique_lons, domain_lon)

# Clip indices to valid ranges
lat_indices = np.clip(lat_indices, 1, len(unique_lats) - 1)
lon_indices = np.clip(lon_indices, 1, len(unique_lons) - 1)

# Get the 4 corner indices for each domain point (vectorized)
lat_ll_idx = lat_indices - 1  # lower-left
lat_ur_idx = lat_indices       # upper-right
lon_ll_idx = lon_indices - 1  # lower-left
lon_ur_idx = lon_indices       # upper-right

# Get actual lat/lon values for corners
lat_ll = unique_lats[lat_ll_idx]
lat_ur = unique_lats[lat_ur_idx]
lon_ll = unique_lons[lon_ll_idx]
lon_ur = unique_lons[lon_ur_idx]

print(f"  ✓ Pre-computed cell indices for all domain points")

# OPTIMIZATION 3: Create efficient lookup structure for rainfall data
# Pivot the dataframe to create a 2D grid for each timestep
print(f"  ✓ Creating efficient rainfall grid lookup...")

# Create mapping from actual lat/lon values to grid indices
lat_to_idx = {lat: i for i, lat in enumerate(unique_lats)}
lon_to_idx = {lon: i for i, lon in enumerate(unique_lons)}

# Pre-compute indices for ALL rainfall data at once (vectorized)
# Using direct mapping for exact matches (not searchsorted)
lat_indices_all = rain_sim['lat'].map(lat_to_idx).values
lon_indices_all = rain_sim['long'].map(lon_to_idx).values

# Get datetime as array for fast comparison
datetime_array = rain_sim['datetime'].values
precip_array = rain_sim['precipitation'].values

rainfall_cells = {}

# Process each timestep
for i, t in enumerate(sim_times):
    # VECTORIZED: Find all records for this timestep
    mask = (datetime_array == t)
    lat_idx_t = lat_indices_all[mask]
    lon_idx_t = lon_indices_all[mask]
    precip_t = precip_array[mask]
    
    # Create 2D array for this timestep (much faster than dict lookup)
    rain_grid = np.zeros((len(unique_lats), len(unique_lons)))
    
    # Vectorized assignment to grid (exact index matching)
    rain_grid[lat_idx_t, lon_idx_t] = precip_t
    
    # VECTORIZED: Get values at 4 corners for ALL domain points at once
    v_ll = rain_grid[lat_ll_idx, lon_ll_idx]  # lower-left
    v_lr = rain_grid[lat_ll_idx, lon_ur_idx]  # lower-right
    v_ul = rain_grid[lat_ur_idx, lon_ll_idx]  # upper-left
    v_ur = rain_grid[lat_ur_idx, lon_ur_idx]  # upper-right
    
    # Average of 4 corners (vectorized) and convert mm/hr to m/s
    rainfall_array = (v_ll + v_lr + v_ul + v_ur) * 0.25 / 3600000.0  # Combined division
    
    # Store using time_seconds (already computed)
    rainfall_cells[time_seconds[i]] = rainfall_array
    
    if i % 100 == 0:  # Progress every 100 hours
        print(f"  Progress: {i+1}/{len(sim_times)} time steps processed")

print(f"\n✓ Created {len(rainfall_cells)} rainfall cell arrays")

# Test at first timestep
print(f"\nTesting cell-based rainfall at first timestep...")
test_rainfall = rainfall_cells[0.0]
print(f"  Min rainfall: {test_rainfall.min()*3600*1000:.6f} mm/hr")
print(f"  Max rainfall: {test_rainfall.max()*3600*1000:.6f} mm/hr")
print(f"  Mean rainfall: {test_rainfall.mean()*3600*1000:.6f} mm/hr")
print(f"  Non-zero points: {np.sum(test_rainfall > 0):,} ({np.sum(test_rainfall > 0)/len(test_rainfall)*100:.1f}%)")

# Additional verification: check the original rainfall grid for first timestep
print(f"\nVerifying original IMERG data at first timestep...")
first_time_data = rain_sim[rain_sim['datetime'] == sim_times[0]]
print(f"  Original IMERG precipitation at t=0:")
print(f"    Min: {first_time_data['precipitation'].min():.6f} mm/hr")
print(f"    Max: {first_time_data['precipitation'].max():.6f} mm/hr")
print(f"    Mean: {first_time_data['precipitation'].mean():.6f} mm/hr")
print(f"    Records: {len(first_time_data)}")
print(f"  After cell-based averaging (4-corner average):")
print(f"    Expected range: slightly different due to averaging")
print(f"    Actual range: {test_rainfall.min()*3600*1000:.6f} to {test_rainfall.max()*3600*1000:.6f} mm/hr")

## 🎯 Create Custom Rainfall Operator Class

In [ ]:
class Spatial_Temporal_Rainfall_Operator(anuga.Rate_operator):
    """
    Custom rainfall operator that applies cell-based constant rainfall values.
    
    Uses ANUGA's Rate_operator which properly handles source terms.
    """
    
    def __init__(self, domain, rainfall_cells, time_seconds, 
                 description=None, label=None, logging=False, verbose=False):
        """
        Initialize the spatial-temporal rainfall operator.
        
        Parameters:
        -----------
        domain : ANUGA Domain
            The computational domain
        rainfall_cells : dict
            Dictionary mapping time (seconds) to rainfall arrays (m/s)
            Each array contains constant rainfall values for each domain triangle
        time_seconds : array
            Array of time values (seconds) for which rainfall data exists
        """
        # Get initial rainfall rate
        self.rainfall_cells = rainfall_cells
        self.time_seconds = time_seconds
        self.n_triangles = domain.get_number_of_triangles()
        self.current_time_idx = -1
        
        # Initialize with first rainfall rate
        initial_rate = rainfall_cells.get(0.0, np.zeros(self.n_triangles))
        
        # Initialize Rate_operator with initial rate
        anuga.Rate_operator.__init__(self, domain, rate=initial_rate,
                        description=description, label=label,
                        logging=logging, verbose=verbose)
        
        print(f"✓ Cell-Based Rainfall Operator initialized")
        print(f"  Domain triangles: {self.n_triangles:,}")
        print(f"  Time steps: {len(self.time_seconds)}")
        print(f"  Method: Constant rainfall per cell (average of 4 grid corners)")
    
    def update_rate(self):
        """
        Update the rainfall rate based on current simulation time.
        This is called automatically by Rate_operator before applying the rate.
        """
        t = self.domain.get_time()
        
        # Find the appropriate rainfall array for current time
        idx = np.searchsorted(self.time_seconds, t)
        
        if idx >= len(self.time_seconds):
            idx = len(self.time_seconds) - 1
        elif idx > 0 and abs(t - self.time_seconds[idx-1]) < abs(t - self.time_seconds[idx]):
            idx = idx - 1
        
        t_key = self.time_seconds[idx]
        
        # Get rainfall array for this time
        if t_key in self.rainfall_cells:
            rainfall_rate = self.rainfall_cells[t_key]  # in m/s
            
            # Ensure non-negative
            rainfall_rate = np.maximum(rainfall_rate, 0.0)
            
            # Update the rate (Rate_operator will apply this)
            self.set_rate(rainfall_rate)
            
            # Update monitoring
            if idx != self.current_time_idx:
                self.current_time_idx = idx
                if self.verbose and idx % 24 == 0:  # Print every 24 hours
                    print(f"  Rainfall updated at t={t/3600:.1f}h: "
                          f"min={rainfall_rate.min()*3600*1000:.3f} mm/hr, "
                          f"max={rainfall_rate.max()*3600*1000:.3f} mm/hr")
        else:
            # No rainfall data, set to zero
            self.set_rate(0.0)
    
    def parallel_safe(self):
        """Required for ANUGA operator framework."""
        return True
    
    def statistics(self):
        """Return statistics about current rainfall."""
        if hasattr(self, 'rate') and self.rate is not None:
            if isinstance(self.rate, np.ndarray):
                rate_array = self.rate
            else:
                rate_array = np.full(self.n_triangles, self.rate)
            
            return {
                'min_rate_mm_hr': float(rate_array.min() * 3600 * 1000),
                'max_rate_mm_hr': float(rate_array.max() * 3600 * 1000),
                'mean_rate_mm_hr': float(rate_array.mean() * 3600 * 1000)
            }
        return {}

print("✓ Custom Cell-Based Rainfall Operator class defined")

## ⚡ Add Rainfall Operator to Domain

In [ ]:
print("\n" + "="*70)
print("ADDING RAINFALL OPERATOR TO DOMAIN")
print("="*70)

# Create and add the cell-based rainfall operator
rainfall_op = Spatial_Temporal_Rainfall_Operator(
    domain,
    rainfall_cells=rainfall_cells,
    time_seconds=time_seconds,
    description="IMERG hourly cell-based rainfall",
    label="IMERG_rainfall",
    verbose=True
)

print("\n✓ Cell-based rainfall operator added to domain")
print("  Each domain triangle receives constant rainfall based on")
print("  the average of 4 surrounding IMERG grid points")

## 🚀 Enable GPU Acceleration

In [ ]:
print("\n" + "="*70)
print("GPU ACCELERATION SETUP")
print("="*70)

if config.use_gpu and GPU_AVAILABLE:
    try:
        # Test GPU
        import cupy as cp
        print("\nTesting GPU accessibility...")
        test_array = cp.random.random((1000, 1000))
        test_result = cp.sum(test_array)
        print(f"✓ GPU computation test passed")
        
        # Enable GPU mode
        print("\nEnabling GPU mode...")
        domain.set_multiprocessor_mode(2)  # 2 = GPU mode
        
        if domain.gpu_interface is not None:
            print("✓ GPU acceleration ENABLED!")
            print(f"  GPU interface: {type(domain.gpu_interface).__name__}")
            print(f"  Multiprocessor mode: {domain.get_multiprocessor_mode()}")
            
            mempool = cp.get_default_memory_pool()
            print(f"\n  GPU Memory:")
            print(f"    Used: {mempool.used_bytes()/1e9:.3f} GB")
            print(f"    Total: {mempool.total_bytes()/1e9:.3f} GB")
            
            print("\n  💡 TIP: Monitor GPU with 'nvidia-smi -l 1' in terminal")
        else:
            raise Exception("GPU interface not created")
            
    except Exception as e:
        print(f"\n✗ GPU setup failed: {e}")
        print("  Falling back to CPU mode...")
        
        import multiprocessing
        num_threads = multiprocessing.cpu_count()
        domain.set_omp_num_threads(num_threads)
        print(f"  ✓ CPU mode: {num_threads} OpenMP threads")
        config.use_gpu = False
else:
    import multiprocessing
    num_threads = multiprocessing.cpu_count()
    domain.set_omp_num_threads(num_threads)
    print(f"✓ CPU mode: {num_threads} OpenMP threads")

print("\n" + "="*70)
print("DOMAIN READY FOR SIMULATION")
print("="*70)
print(f"Triangles: {domain.get_number_of_triangles():,}")
print(f"Area: {domain.get_area()/1e6:.2f} km²")
print(f"Mode: {'🔥 GPU (CUDA/CuPy)' if config.use_gpu else '🔧 CPU (OpenMP)'}")
print("="*70)

## 📈 Setup Monitoring and Checkpointing

In [ ]:
# Monitoring arrays
time_series = []
max_depth_series = []
mean_depth_series = []
total_volume_series = []
timestep_series = []

# Checkpointing setup
checkpoint_counter = 0
last_checkpoint_time = 0

def save_checkpoint():
    """Save simulation state to checkpoint file."""
    global checkpoint_counter
    
    checkpoint_file = os.path.join(config.output_dir, f'checkpoint_{checkpoint_counter:03d}.pkl')
    
    checkpoint_data = {
        'time': domain.get_time(),
        'quantities': {
            'stage': domain.get_quantity('stage').centroid_values.copy(),
            'xmomentum': domain.get_quantity('xmomentum').centroid_values.copy(),
            'ymomentum': domain.get_quantity('ymomentum').centroid_values.copy(),
        },
        'timeseries': {
            'time': np.array(time_series),
            'max_depth': np.array(max_depth_series),
            'mean_depth': np.array(mean_depth_series),
            'total_volume': np.array(total_volume_series),
            'timestep': np.array(timestep_series)
        }
    }
    
    import pickle
    with open(checkpoint_file, 'wb') as f:
        pickle.dump(checkpoint_data, f)
    
    print(f"\n  ✓ Checkpoint saved: {checkpoint_file}")
    checkpoint_counter += 1

print("✓ Monitoring and checkpointing configured")

## 🎬 Run Simulation

In [ ]:
print("\n" + "="*70)
print("STARTING SIMULATION")
print("="*70)

# Simulation parameters
yieldstep = config.output_interval_hours * 3600  # Convert to seconds
finaltime = (config.end_date - config.start_date).total_seconds()
checkpoint_interval = config.checkpoint_interval_hours * 3600  # Convert to seconds

print(f"\nSimulation parameters:")
print(f"  Duration: {finaltime/3600:.1f} hours ({finaltime/86400:.1f} days)")
print(f"  Output interval: {yieldstep/3600:.1f} hours")
print(f"  Checkpoint interval: {checkpoint_interval/3600:.1f} hours")

# Start simulation
print(f"\n{'='*70}")
print("SIMULATION RUNNING...")
print(f"{'='*70}\n")

sim_start = time.time()
step_count = 0

try:
    # Pre-compute elevation and areas (they don't change)
    elevation = domain.get_quantity('elevation').centroid_values
    areas = domain.areas
    
    for t in domain.evolve(yieldstep=yieldstep, finaltime=finaltime):
        # Get quantities (stage is the only changing quantity)
        stage = domain.get_quantity('stage').centroid_values
        
        # Vectorized depth calculation
        depth = np.maximum(stage - elevation, 0.0)
        
        # Compute statistics (fully vectorized)
        max_depth = depth.max()
        wet_mask = depth > 0.001
        mean_depth = depth[wet_mask].mean() if wet_mask.any() else 0.0
        total_volume = (depth * areas).sum()
        current_timestep = domain.get_timestep()
        
        # Store timeseries
        time_series.append(t)
        max_depth_series.append(max_depth)
        mean_depth_series.append(mean_depth)
        total_volume_series.append(total_volume)
        timestep_series.append(current_timestep)
        
        # Progress report
        step_count += 1
        elapsed = time.time() - sim_start
        progress = (t / finaltime) * 100
        eta = (elapsed / t) * (finaltime - t) if t > 0 else 0
        
        print(f"Time: {t/3600:.1f}h / {finaltime/3600:.1f}h ({progress:.1f}%) | "
              f"Max depth: {max_depth:.5f}m | "
              f"Mean depth: {mean_depth:.5f}m | "
              f"Volume: {total_volume:.4f}m³ | "
              f"dt: {current_timestep:.5f}s | "
              f"ETA: {eta/60:.0f}min")
        
        # Checkpoint
        if config.save_checkpoints and (t - last_checkpoint_time) >= checkpoint_interval:
            save_checkpoint()
            last_checkpoint_time = t
            
            # Save timeseries data
            np.savez(os.path.join(config.output_dir, 'timeseries_data.npz'),
                    time=np.array(time_series),
                    max_depth=np.array(max_depth_series),
                    mean_depth=np.array(mean_depth_series),
                    total_volume=np.array(total_volume_series),
                    timestep=np.array(timestep_series))
    
    # Final save
    save_checkpoint()
    np.savez(os.path.join(config.output_dir, 'timeseries_data.npz'),
            time=np.array(time_series),
            max_depth=np.array(max_depth_series),
            mean_depth=np.array(mean_depth_series),
            total_volume=np.array(total_volume_series),
            timestep=np.array(timestep_series))
    
    total_time = time.time() - sim_start
    
    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE!")
    print(f"{'='*70}")
    print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
    print(f"Time steps: {step_count}")
    print(f"Average time per step: {total_time/step_count:.2f} seconds")
    print(f"\nResults saved to: {config.output_dir}/")
    print(f"  - {config.simulation_name}.sww (full spatial results)")
    print(f"  - timeseries_data.npz (time series data)")
    print(f"  - checkpoint_*.pkl (checkpoints)")
    
except KeyboardInterrupt:
    print("\n\n⚠ SIMULATION INTERRUPTED BY USER")
    print("Saving current state...")
    save_checkpoint()
    np.savez(os.path.join(config.output_dir, 'timeseries_data.npz'),
            time=np.array(time_series),
            max_depth=np.array(max_depth_series),
            mean_depth=np.array(mean_depth_series),
            total_volume=np.array(total_volume_series),
            timestep=np.array(timestep_series))
    print("✓ State saved. You can analyze partial results.")

except Exception as e:
    print(f"\n\n✗ ERROR: {e}")
    import traceback
    traceback.print_exc()
    print("\nAttempting to save current state...")
    try:
        save_checkpoint()
    except:
        print("Failed to save checkpoint")

## 📊 Visualize Results

In [ ]:
print("\n" + "="*70)
print("VISUALIZING RESULTS")
print("="*70)

# Load timeseries data
data = np.load(os.path.join(config.output_dir, 'timeseries_data.npz'))
time_hrs = data['time'] / 3600.0
max_depth = data['max_depth']
mean_depth = data['mean_depth']
total_volume = data['total_volume']
timestep = data['timestep']

# Create plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Max depth
ax1 = axes[0, 0]
ax1.plot(time_hrs, max_depth, 'b-', linewidth=2)
ax1.set_xlabel('Time (hours)', fontsize=12)
ax1.set_ylabel('Maximum Depth (m)', fontsize=12)
ax1.set_title('Maximum Water Depth Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Mean depth
ax2 = axes[0, 1]
ax2.plot(time_hrs, mean_depth, 'g-', linewidth=2)
ax2.set_xlabel('Time (hours)', fontsize=12)
ax2.set_ylabel('Mean Depth (m)', fontsize=12)
ax2.set_title('Mean Water Depth Over Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Total volume
ax3 = axes[1, 0]
ax3.plot(time_hrs, total_volume, 'r-', linewidth=2)
ax3.set_xlabel('Time (hours)', fontsize=12)
ax3.set_ylabel('Total Volume (m³)', fontsize=12)
ax3.set_title('Total Water Volume Over Time', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Timestep
ax4 = axes[1, 1]
ax4.plot(time_hrs, timestep, 'purple', linewidth=2)
ax4.set_xlabel('Time (hours)', fontsize=12)
ax4.set_ylabel('Timestep (s)', fontsize=12)
ax4.set_title('Adaptive Timestep Over Time', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
output_file = os.path.join(config.output_dir, 'simulation_results.png')
plt.savefig(output_file, dpi=150, bbox_inches='tight')
print(f"\n✓ Results plot saved: {output_file}")
plt.show()

# Print summary statistics
print(f"\n{'='*70}")
print("SIMULATION SUMMARY")
print(f"{'='*70}")
print(f"Maximum depth reached: {np.max(max_depth):.3f} m")
print(f"Maximum volume: {np.max(total_volume):.1f} m³")
print(f"Mean depth (overall): {np.mean(mean_depth):.3f} m")
print(f"Average timestep: {np.mean(timestep):.3f} s")
print(f"Min timestep: {np.min(timestep):.3f} s")
print(f"Max timestep: {np.max(timestep):.3f} s")
print(f"{'='*70}")

## 🎥 Create Animation (Optional)

In [ ]:
def create_animation(quantity='depth', frames=100, fps=10):
    """
    Create animation from SWW file.
    
    Parameters:
    -----------
    quantity : str
        Quantity to visualize ('depth', 'stage', 'speed', etc.)
    frames : int
        Number of frames to extract
    fps : int
        Frames per second
    """
    from matplotlib.animation import FuncAnimation, PillowWriter
    from matplotlib.tri import Triangulation
    from anuga.file.netcdf import NetCDFFile
    
    print(f"\nCreating animation for {quantity}...")
    
    sww_file = os.path.join(config.output_dir, f'{config.simulation_name}.sww')
    if not os.path.exists(sww_file):
        print(f"✗ SWW file not found: {sww_file}")
        return
    
    # Open SWW file
    fid = NetCDFFile(sww_file, 'r')
    
    # Get mesh
    x = fid.variables['x'][:]
    y = fid.variables['y'][:]
    volumes = fid.variables['volumes'][:]
    time = fid.variables['time'][:]
    
    tri = Triangulation(x, y, volumes)
    
    # Get data (all operations vectorized)
    if quantity == 'depth':
        stage = fid.variables['stage'][:]
        elevation = fid.variables['elevation'][:]
        data = np.maximum(stage - elevation, 0.0)  # Vectorized
        label = 'Water Depth (m)'
        cmap = 'Blues'
    elif quantity == 'stage':
        data = fid.variables['stage'][:]
        label = 'Water Stage (m)'
        cmap = 'viridis'
    elif quantity == 'speed':
        xmom = fid.variables['xmomentum'][:]
        ymom = fid.variables['ymomentum'][:]
        stage = fid.variables['stage'][:]
        elevation = fid.variables['elevation'][:]
        depth = np.maximum(stage - elevation, 0.001)
        # Vectorized velocity and speed calculation
        u = np.divide(xmom, depth, where=depth>0.001, out=np.zeros_like(xmom))
        v = np.divide(ymom, depth, where=depth>0.001, out=np.zeros_like(ymom))
        data = np.hypot(u, v)  # Faster than sqrt(u**2 + v**2)
        label = 'Flow Speed (m/s)'
        cmap = 'Reds'
    
    fid.close()
    
    # Select frames
    n_times = len(time)
    frame_indices = np.linspace(0, n_times-1, min(frames, n_times), dtype=int)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Initial plot
    vmin = np.percentile(data, 1)
    vmax = np.percentile(data, 99)
    
    tpc = ax.tripcolor(tri, data[0], cmap=cmap, vmin=vmin, vmax=vmax, shading='flat')
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (m)', fontsize=12)
    ax.set_ylabel('Northing (m)', fontsize=12)
    title = ax.set_title(f'{label} at t=0.0 hours', fontsize=14, fontweight='bold')
    cbar = plt.colorbar(tpc, ax=ax, label=label)
    
    def update(frame):
        idx = frame_indices[frame]
        tpc.set_array(data[idx])
        title.set_text(f'{label} at t={time[idx]/3600:.1f} hours')
        return tpc, title
    
    anim = FuncAnimation(fig, update, frames=len(frame_indices), interval=1000/fps, blit=False)
    
    # Save
    output_file = os.path.join(config.output_dir, f'{quantity}_animation.gif')
    writer = PillowWriter(fps=fps)
    anim.save(output_file, writer=writer)
    
    print(f"✓ Animation saved: {output_file}")
    plt.close()

# Create animations
print("\n" + "="*70)
print("CREATING ANIMATIONS")
print("="*70)

try:
    create_animation(quantity='depth', frames=50, fps=5)
    create_animation(quantity='speed', frames=50, fps=5)
    print("\n✓ Animations created successfully!")
except Exception as e:
    print(f"\n✗ Animation creation failed: {e}")
    print("  (This is optional - main results are still available)")

: 

## ✅ Simulation Complete!

Your rainfall simulation is complete. The results include:

1. **SWW file**: Full spatial-temporal data for detailed analysis
2. **Timeseries data**: NPZ file with depth, volume, and timestep data
3. **Plots**: PNG images showing simulation progress
4. **Animations**: GIF files showing flow evolution
5. **Checkpoints**: PKL files for recovery if needed

All files are saved in the `outputs/` directory.